## Feature Preparation for Structural Phase Classification

This notebook prepares composition-based descriptors for the
`Multi-phase` classification label in the structural information module.

### Workflow

1. Load compound formulas, phase labels, and elemental descriptors.
2. Weight elemental descriptors by stoichiometric coefficients.
3. Sum the metal contributions and append the nonmetal descriptors.
4. Export 25 features with the target column `label`.

Formulas must contain one or two metals followed by one nonmetal.
Descriptor weighting uses the formula coefficients without normalization.
Run cells sequentially; the final export includes the labels.

In [2]:
# ============================================================
# 1. Import dependencies
# ============================================================

import re
import json

import numpy as np
import pandas as pd

In [3]:
# ============================================================
# 2. Load compounds, labels, and elemental descriptors
# ============================================================

# Select rows with index labels 0 through 165, inclusive.
tmcs = pd.read_excel(
    "data/6_Structural_label.xlsx",
    sheet_name="TMCs",
    usecols=["compounds", "Multi-phase"],
).loc[:165, :]

print("Compound table shape:", tmcs.shape)

# Load 13 metal descriptors in the specified column order.
cols_tm = [
    "element",
    "AN",
    "AW",
    "CR",
    "AR",
    "FIE",
    "SIE",
    "EA",
    "EN",
    "Nd",
    "Nv",
    "D",
    "EC",
    "TC",
]

# Replace missing values and "/" placeholders with zero.
tm = (
    pd.read_excel(
        "data/6_Structural_label.xlsx",
        sheet_name="TM_descriptors",
        usecols=cols_tm,
    )
    .fillna(0, inplace=False)
    .replace("/", 0, inplace=False)
)

# Load 12 nonmetal descriptors.
cols_c = [
    "element",
    "AN",
    "AW",
    "CR",
    "AR",
    "VR",
    "FIE",
    "SIE",
    "EA",
    "EN",
    "Np",
    "Nv",
    "D",
]

# Replace missing nonmetal descriptor values with zero.
c = (
    pd.read_excel(
        "data/6_Structural_label.xlsx",
        sheet_name="C_descriptors",
        usecols=cols_c,
    )
    .fillna(0, inplace=False)
)

Compound table shape: (160, 2)


In [4]:
# ============================================================
# 3. Define chemical formula parsing functions
# ============================================================

def parse_compound(compound):
    """Extract element symbols and coefficients from a simple formula.

    Parameters
    ----------
    compound : str
        Chemical formula containing element symbols and optional
        integer or decimal stoichiometric coefficients.

    Returns
    -------
    dict
        Element symbols mapped to their stoichiometric coefficients,
        preserving their order of appearance.

    Notes
    -----
    An omitted coefficient is interpreted as 1.0.
    This parser is intended for simple formulas without parentheses.
    Repeated element symbols overwrite earlier values.
    """
    matches = re.findall(
        r"([A-Z][a-z]*)(\d*\.*\d*)",
        compound,
    )

    result = {}

    for match in matches:
        element, count = match

        # Use a coefficient of 1.0 when no number is provided.
        count = float(count) if count else 1.0
        result[element] = count

    return result


def get_res(compound_name):
    """Return the parsed elemental composition of a compound formula."""
    parsed_result = parse_compound(compound_name)
    return parsed_result


# Check formula parsing using a representative bimetallic nitride.
test = get_res("Mn0.2Cr0.8N")

In [5]:
# ============================================================
# 4. Inspect descriptor columns and preliminary column templates
# ============================================================

# Inspect the descriptor columns available in each input table.
display(tm.columns.tolist())
display(c.columns.tolist())

# Retain the original preliminary column template for reference.
# This template is not used to construct the final classification table.
columns = (
    [f"AB_{i}" for i in tm.columns.tolist()[1:]]
    + [f"C_{i}" for i in c.columns.tolist()[1:]]
    + [
        "a_1",
        "b_1",
        "c_1",
        "Grain_1",
        "wt_1",
        "a_2",
        "b_2",
        "c_2",
        "Grain_2",
        "Multi-phase",
        "CurrentDensity",
        "overpotential",
    ]
)

display(np.array(columns))

# Retain the original alternative column template.
# These separate metal feature blocks are not used in the final export.
col = ["compound"]

# Define 14 placeholder columns for the first metal.
metal1_cols = [
    f"metal1_feat_{i}"
    for i in range(1, 15)
]

# Define 14 placeholder columns for the second metal.
metal2_cols = [
    f"metal2_feat_{i}"
    for i in range(1, 15)
]

# Define 12 placeholder columns for the nonmetal.
nonmetal_cols = [
    f"nonmetal_feat_{i}"
    for i in range(1, 13)
]

# Define the target column.
tail = ["label"]

feature_cols = (
    metal1_cols
    + metal2_cols
    + nonmetal_cols
)

all_columns = col + feature_cols + tail

print(
    "Compound column data type:",
    tmcs["compounds"].dtype,
)


['element',
 'AN',
 'AW',
 'CR',
 'AR',
 'FIE',
 'SIE',
 'EA',
 'EN',
 'Nd',
 'Nv',
 'D',
 'EC',
 'TC']

['element',
 'AN',
 'AW',
 'CR',
 'AR',
 'VR',
 'FIE',
 'SIE',
 'EA',
 'EN',
 'Np',
 'Nv',
 'D']

array(['AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE', 'AB_EA',
       'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN',
       'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN',
       'C_Np', 'C_Nv', 'C_D', 'a_1', 'b_1', 'c_1', 'Grain_1', 'wt_1',
       'a_2', 'b_2', 'c_2', 'Grain_2', 'Multi-phase', 'CurrentDensity',
       'overpotential'], dtype='<U14')

Compound column data type: object


In [6]:
# ============================================================
# 5. Generate the preliminary feature table without labels
# ============================================================

train_data = []

# Align positional iteration with subsequent label-based row access.
tmcs = tmcs.reset_index(drop=True)

for index, formula in enumerate(tmcs["compounds"]):

    compound_feat = [formula]

    composition = get_res(formula)

    keys = list(composition.keys())
    values = list(composition.values())

    # Expected order for a ternary formula:
    # first metal, second metal, and nonmetal.
    if len(keys) > 2:

        A_key = keys[0]
        B_key = keys[1]
        C_key = keys[2]

        A_index = float(values[0])
        B_index = float(values[1])
        C_index = float(values[2])

        # Weight each metal descriptor vector by its formula coefficient.
        A_feat = (
            tm[tm["element"] == A_key]
            .values
            .flatten()[1:]
            .astype(float)
            * A_index
        )

        B_feat = (
            tm[tm["element"] == B_key]
            .values
            .flatten()[1:]
            .astype(float)
            * B_index
        )

        # Weight the nonmetal descriptors by the nonmetal coefficient.
        C_feat = (
            c[c["element"] == C_key]
            .values
            .flatten()[1:]
            .astype(float)
            * C_index
        )

        # Sum metal contributions without normalizing their coefficients.
        AB_feat = A_feat + B_feat

    else:

        # Expected order for a binary formula: metal and nonmetal.
        AB_key = keys[0]
        C_key = keys[1]

        AB_index = float(values[0])
        C_index = float(values[1])

        AB_feat = (
            tm[tm["element"] == AB_key]
            .values
            .flatten()[1:]
            .astype(float)
            * AB_index
        )

        C_feat = (
            c[c["element"] == C_key]
            .values
            .flatten()[1:]
            .astype(float)
            * C_index
        )

    # Concatenate the metal and nonmetal descriptor blocks.
    # Current density and overpotential are excluded from this task.
    compound_feat += (
        list(AB_feat)
        + list(C_feat)
        # + [
        #     tmcs.loc[index, "CurrentDensity"],
        #     tmcs.loc[index, "overpotential"],
        # ]
    )

    train_data.append(compound_feat)

columns = (
    [f"AB_{column}" for column in tm.columns[1:]]
    + [f"C_{column}" for column in c.columns[1:]]
    # + [
    #     "CurrentDensity",
    #     "overpotential",
    # ]
)

colnames = ["compound"] + columns

train_data_ = pd.DataFrame(
    train_data,
    columns=colnames,
)

# This preliminary export is overwritten by the labeled table in Section 6.
train_data_.to_csv(
    "data/compound_6_Structural_label.csv",
    index=False,
)

print("Data shape:", train_data_.shape)

# Exclude only the compound identifier because labels are not yet included.
print("Feature number:", train_data_.shape[1] - 1)

display(train_data_.head())


Data shape: (160, 26)
Feature number: 25


,compound,AB_AN,AB_AW,AB_CR,AB_AR,AB_FIE,AB_SIE,AB_EA,AB_EN,AB_Nd,...,C_CR,C_AR,C_VR,C_FIE,C_SIE,C_EA,C_EN,C_Np,C_Nv,C_D
0,CoS2,27.0,58.933194,1.260,1.520,7.8810,17.0800,0.6600,1.880,7.0,...,2.1,1.76,3.6,20.72,46.68,4.14,5.16,8.0,12.0,3.92
1,Cr0.1Co0.9S2,26.7,58.239485,1.261,1.534,7.7696,16.9908,0.6606,1.858,6.7,...,2.1,1.76,3.6,20.72,46.68,4.14,5.16,8.0,12.0,3.92
2,Cr0.2Co0.8S2,26.4,57.545775,1.262,1.548,7.6582,16.9016,0.6612,1.836,6.4,...,2.1,1.76,3.6,20.72,46.68,4.14,5.16,8.0,12.0,3.92
3,Cr0.5Co0.5S2,25.5,55.464647,1.265,1.590,7.3240,16.6340,0.6630,1.770,5.5,...,2.1,1.76,3.6,20.72,46.68,4.14,5.16,8.0,12.0,3.92
4,Fe0.1Co0.9S2,26.9,58.624375,1.259,1.524,7.8832,17.1889,0.6103,1.875,6.9,...,2.1,1.76,3.6,20.72,46.68,4.14,5.16,8.0,12.0,3.92


In [7]:
# ============================================================
# 6. Generate the final feature table with classification labels
# ============================================================

train_data = []

tmcs = tmcs.reset_index(drop=True)

for index, formula in enumerate(tmcs["compounds"]):

    compound_feat = [formula]

    composition = get_res(formula)

    keys = list(composition.keys())
    values = list(composition.values())

    # Process compounds containing two metals followed by one nonmetal.
    if len(keys) > 2:

        A_key = keys[0]
        B_key = keys[1]
        C_key = keys[2]

        A_index = float(values[0])
        B_index = float(values[1])
        C_index = float(values[2])

        # Retrieve and weight descriptors for the first metal.
        A_feat = (
            tm[tm["element"] == A_key]
            .values
            .flatten()[1:]
            .astype(float)
            * A_index
        )

        # Retrieve and weight descriptors for the second metal.
        B_feat = (
            tm[tm["element"] == B_key]
            .values
            .flatten()[1:]
            .astype(float)
            * B_index
        )

        # Retrieve and weight descriptors for the nonmetal.
        C_feat = (
            c[c["element"] == C_key]
            .values
            .flatten()[1:]
            .astype(float)
            * C_index
        )

        # Combine both metals into a single 13-descriptor block.
        AB_feat = A_feat + B_feat

    else:

        # Process compounds containing one metal followed by one nonmetal.
        AB_key = keys[0]
        C_key = keys[1]

        AB_index = float(values[0])
        C_index = float(values[1])

        AB_feat = (
            tm[tm["element"] == AB_key]
            .values
            .flatten()[1:]
            .astype(float)
            * AB_index
        )

        C_feat = (
            c[c["element"] == C_key]
            .values
            .flatten()[1:]
            .astype(float)
            * C_index
        )

    # Append the original phase label without modifying its encoding.
    compound_feat += (
        list(AB_feat)
        + list(C_feat)
        + [
            tmcs.loc[index, "Multi-phase"],
        ]
    )

    train_data.append(compound_feat)

# Use descriptor names to identify the metal and nonmetal feature blocks.
columns = (
    [f"AB_{column}" for column in tm.columns[1:]]
    + [f"C_{column}" for column in c.columns[1:]]
    + ["label"]
)

colnames = ["compound"] + columns

train_data_ = pd.DataFrame(
    train_data,
    columns=colnames,
)

# Export the final table, replacing the preliminary feature-only CSV.
train_data_.to_csv(
    "data/compound_6_Structural_label.csv",
    index=False,
)

print("Data shape:", train_data_.shape)

# Exclude the compound identifier and the target label from the count.
print("Feature number:", train_data_.shape[1] - 2)

display(train_data_.head())


Data shape: (160, 27)
Feature number: 25


,compound,AB_AN,AB_AW,AB_CR,AB_AR,AB_FIE,AB_SIE,AB_EA,AB_EN,AB_Nd,...,C_AR,C_VR,C_FIE,C_SIE,C_EA,C_EN,C_Np,C_Nv,C_D,label
0,CoS2,27.0,58.933194,1.260,1.520,7.8810,17.0800,0.6600,1.880,7.0,...,1.76,3.6,20.72,46.68,4.14,5.16,8.0,12.0,3.92,0
1,Cr0.1Co0.9S2,26.7,58.239485,1.261,1.534,7.7696,16.9908,0.6606,1.858,6.7,...,1.76,3.6,20.72,46.68,4.14,5.16,8.0,12.0,3.92,0
2,Cr0.2Co0.8S2,26.4,57.545775,1.262,1.548,7.6582,16.9016,0.6612,1.836,6.4,...,1.76,3.6,20.72,46.68,4.14,5.16,8.0,12.0,3.92,0
3,Cr0.5Co0.5S2,25.5,55.464647,1.265,1.590,7.3240,16.6340,0.6630,1.770,5.5,...,1.76,3.6,20.72,46.68,4.14,5.16,8.0,12.0,3.92,0
4,Fe0.1Co0.9S2,26.9,58.624375,1.259,1.524,7.8832,17.1889,0.6103,1.875,6.9,...,1.76,3.6,20.72,46.68,4.14,5.16,8.0,12.0,3.92,0


In [8]:
# ============================================================
# 7. Inspect the final output columns
# ============================================================

display(train_data_.columns)

Index(['compound', 'AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE',
       'AB_EA', 'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN',
       'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN',
       'C_Np', 'C_Nv', 'C_D', 'label'],
      dtype='object')